In [3]:
#!/usr/bin/env python3
"""
Pure-stdlib solution to get the VIAF MARC-21 record for a given VIAF id.

Two attempts (in order):
  A) Fetch the VIAF page HTML and parse anchors/buttons for "MARC-21 RECORD".
     If we find an href, download that URL.
  B) If A fails (likely because link is client-rendered by JS), stream the
     VIAF MARC-XML dump and search for the VIAF id. (Requires network and
     downloads a multi-GB file stream but reads one record at a time.)

No external packages required.
"""

import sys
import urllib.request
import urllib.parse
import urllib.error
from html.parser import HTMLParser
import gzip
import io
import time

VIAF_PAGE = "https://viaf.org/en/viaf/120307688"
VIAF_ID = "120307688"
# Bulk dump (example found on VIAF data page; update if VIAF publishes a newer dump)
MARCXML_DUMP = "https://downloads.viaf.org/download/yearly/viaf-20240804-clusters-rdf.nt.gz"

OUTFILE = "viaf_{}_marc21.xml".format(VIAF_ID)

# Simple HTML parser to find anchors or elements with the exact visible text.
class MarcLinkParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.candidate_hrefs = []   # list of (href, text)
        self._current_tag = None
        self._current_attrs = None
        self._text_parts = []

    def handle_starttag(self, tag, attrs):
        self._current_tag = tag
        self._current_attrs = dict(attrs)
        self._text_parts = []

    def handle_endtag(self, tag):
        if self._current_tag and tag == self._current_tag:
            visible = "".join(self._text_parts).strip()
            href = self._current_attrs.get("href")
            if visible and "MARC-21" in visible.upper():
                # record candidate even if href is None (button may be JS)
                self.candidate_hrefs.append((href, visible))
            # reset
            self._current_tag = None
            self._current_attrs = None
            self._text_parts = []

    def handle_data(self, data):
        if self._current_tag:
            self._text_parts.append(data)

def fetch_url(url, timeout=30):
    req = urllib.request.Request(url, headers={
        "User-Agent": "python-urllib/3 (viaf-marc-fetcher)"
    })
    return urllib.request.urlopen(req, timeout=timeout)

def attempt_page_scrape(viaf_page):
    print("Attempt A: fetching page HTML and looking for 'MARC-21' link...")
    try:
        with fetch_url(viaf_page) as resp:
            content_type = resp.headers.get("Content-Type", "")
            # Read a reasonable chunk (not necessarily whole page, but pages are small)
            raw = resp.read().decode(resp.headers.get_content_charset("utf-8"), errors="replace")
    except Exception as e:
        print("  Failed to fetch page HTML:", e)
        return None

    parser = MarcLinkParser()
    parser.feed(raw)

    # Prefer first candidate that has an href
    for href, text in parser.candidate_hrefs:
        if href:
            full = urllib.parse.urljoin(viaf_page, href)
            print("  Found candidate href in HTML:", full, " (text:", text, ")")
            return full

    # If candidates exist but no href (a button that triggers JS), we can't click it w/out a browser;
    if parser.candidate_hrefs:
        print("  Found element(s) with 'MARC-21' text, but no href attribute (likely JS).")
        return None

    print("  No server-side MARC-21 link found in fetched HTML.")
    return None

def download_file(url, outpath):
    print("Downloading from:", url)
    try:
        with fetch_url(url) as r:
            with open(outpath, "wb") as f:
                chunk = r.read(8192)
                while chunk:
                    f.write(chunk)
                    chunk = r.read(8192)
        print("Saved to", outpath)
        return outpath
    except Exception as e:
        print("  Download failed:", e)
        return None

def stream_search_marcxml_dump(dump_url, viaf_id, outpath=None):
    """Stream the gzipped MARCXML dump and find the line containing the VIAF id.
       Each line in this dump is expected to be a MARCXML record.
       Returns the record text (string) or None.
    """
    print("Attempt B: streaming MARCXML dump and searching for VIAF id", viaf_id)
    target = viaf_id.encode("utf-8")
    req = urllib.request.Request(dump_url, headers={
        "User-Agent": "python-urllib/3 (viaf-marc-fetcher)"
    })
    try:
        with urllib.request.urlopen(req, timeout=60) as resp:
            # wrap in GzipFile to decompress on the fly
            with gzip.GzipFile(fileobj=resp) as gz:
                # iterate by line (records are one per line)
                # use buffered reading to avoid reading huge chunks
                for i, raw_line in enumerate(gz):
                    # raw_line is bytes
                    if target in raw_line:
                        try:
                            rec_text = raw_line.decode("utf-8", errors="replace")
                        except Exception:
                            rec_text = raw_line.decode("latin1", errors="replace")
                        if outpath:
                            with open(outpath, "w", encoding="utf-8") as f:
                                f.write(rec_text)
                            print("Wrote matching MARCXML record to", outpath)
                            return outpath
                        print("Found matching record in dump (record #{})".format(i+1))
                        return rec_text
                    # optional: some progress indicator every N lines
                    if (i+1) % 250000 == 0:
                        print("  scanned {} records...".format(i+1))
    except Exception as e:
        print("  Error streaming dump:", e)
        return None

    print("  Did not find VIAF id in dump stream.")
    return None

def main():
    # 1) Try scraping page HTML for link
    href = attempt_page_scrape(VIAF_PAGE)
    if href:
        # Download the file at href (may be xml or a download endpoint)
        res = download_file(href, OUTFILE)
        if res:
            print("Done via page-scrape path.")
            return

    # 2) Fall back to streaming the MARCXML dump (no installs)
    res = stream_search_marcxml_dump(MARCXML_DUMP, VIAF_ID, outpath=OUTFILE)
    if res:
        print("Done via dump-stream path.")
        return

    print("Both attempts failed. If the page link is generated only by JavaScript,")
    print("a browser automation (Selenium/Playwright) is required to click it.")

if __name__ == "__main__":
    main()

Attempt A: fetching page HTML and looking for 'MARC-21' link...
  Found element(s) with 'MARC-21' text, but no href attribute (likely JS).
Attempt B: streaming MARCXML dump and searching for VIAF id 120307688
  scanned 250000 records...
  scanned 500000 records...
  scanned 750000 records...
  scanned 1000000 records...
  scanned 1250000 records...
  scanned 1500000 records...
  scanned 1750000 records...
  scanned 2000000 records...
  scanned 2250000 records...
  scanned 2500000 records...
  scanned 2750000 records...
  scanned 3000000 records...
  scanned 3250000 records...
  scanned 3500000 records...
  scanned 3750000 records...
  scanned 4000000 records...
  scanned 4250000 records...
  scanned 4500000 records...
  scanned 4750000 records...
  scanned 5000000 records...
  scanned 5250000 records...
  scanned 5500000 records...
  scanned 5750000 records...
  scanned 6000000 records...
  scanned 6250000 records...
  scanned 6500000 records...
  scanned 6750000 records...
  scanned 7

In [6]:
python viaf_marc_fetch.py 120307688

SyntaxError: invalid syntax (920422033.py, line 1)